In [1]:
import urllib.request
import json
from datetime import datetime

stations = ["GDN_01", "GDN_02", "SOP_01", "GBA_01"]
all_measurements = []

for station in stations:
    url = f"https://e6uw49pbah.execute-api.us-east-1.amazonaws.com/dev/weather/batch?station_id={station}&limit=100"
    req = urllib.request.Request(url)
    req.add_header("Authorization", "Bearer STUDENT_TOKEN_2026")
    
    try:
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                data_string = response.read().decode('utf-8')
                weather_data = json.loads(data_string)
                records = weather_data.get('records', [])
                all_measurements.extend(records)
    except Exception:
        pass

if all_measurements:
    rdd = spark.sparkContext.parallelize([json.dumps(record) for record in all_measurements])
    df_raw = spark.read.json(rdd)
    
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_s3_path = f"s3://michal-super-bucket-6767/raw_data/weather_multi_station_{timestamp_str}.json"
    df_raw.write.mode("overwrite").json(raw_s3_path)

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1781406875087_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
from pyspark.sql import functions as F

s3_raw_path = "s3://michal-super-bucket-6767/raw_data/weather_multi_station_*.json"
df_raw = spark.read.json(s3_raw_path)

df_clean = df_raw.withColumn("temperature", F.col("temperature").cast("double")) \
                 .withColumn("wind_speed", F.col("wind_speed").cast("double")) \
                 .withColumn("rain_mm", F.col("rain_mm").cast("double")) \
                 .withColumn("cloud_cover", F.col("cloud_cover").cast("double")) \
                 .dropna(subset=["temperature", "wind_speed", "rain_mm", "cloud_cover"])

df_labeled = df_clean.withColumn(
    "activity",
    F.when((F.col("rain_mm") > 3.0) & (F.col("temperature") < 10), "Kino")
     .when((F.col("rain_mm") > 3.0) & (F.col("temperature") >= 10), "Basen kryty")
     .when((F.col("rain_mm") > 0.0) & (F.col("rain_mm") <= 3.0), "Muzeum")
     .when((F.col("rain_mm") == 0.0) & (F.col("wind_speed") > 10.0), "Kitesurfing")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature") > 25.0), "Plazing")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature").between(15, 25)), "Jazda na rowerze")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature").between(0, 15)), "Spacer")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature") < 0), "Morsowanie")
     .otherwise("Silownia")
)

curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
df_labeled.write.mode("overwrite").parquet(curated_s3_path)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
df_input = spark.read.parquet(curated_s3_path)
train_data, test_data = df_input.randomSplit([0.7, 0.3], seed=42)

indexer = StringIndexer(inputCol="activity", outputCol="label", handleInvalid="skip")
assembler = VectorAssembler(inputCols=["temperature", "wind_speed", "rain_mm", "cloud_cover"], outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20)

pipeline = Pipeline(stages=[indexer, assembler, rf])
trained_model = pipeline.fit(train_data)

predictions = trained_model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Dokladnosc modelu: {accuracy * 100:.2f}%\n")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Dokladnosc modelu: 98.77%

In [4]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, ArrayType, StringType

labels_list = trained_model.stages[0].labels

explanations = {
    "Kino": "Idealne na mocny deszcz i chlod.",
    "Basen kryty": "Dobra opcja, gdy mocno pada, ale jest w miare cieplo.",
    "Muzeum": "Lekki deszcz nie przeszkadza w zwiedzaniu pod dachem.",
    "Kitesurfing": "Swietne warunki wietrzne i brak opadow.",
    "Plazowanie": "Wysoka temperatura i brak deszczu - idealnie na zewnatrz.",
    "Jazda na rowerze": "Optymalna temperatura na rekreacje na zewnatrz.",
    "Spacer": "Rzesko i bez opadow, w sam raz na spacer.",
    "Morsowanie": "Ujemne temperatury to idealne warunki dla morsow.",
    "Silownia": "Uniwersalna aktywnosc pod dachem na przejsciowa pogode."
}

lokalizacje_stacji = {
    "GDN_01": "Gdańsk Śródmieście",
    "GDN_02": "Gdańsk Oliwa",
    "SOP_01": "Sopot Molo",
    "GBA_01": "Gdynia Orłowo"
}

def get_top_3_with_explanation(prob_vector):
    probs = prob_vector.toArray()
    scored = zip(labels_list, probs)
    sorted_scored = sorted(scored, key=lambda x: x[1], reverse=True)
    
    wyniki = []
    for act, prob in sorted_scored[:3]:
        reason = explanations.get(act, "Odpowiednie warunki pogodowe.")
        wyniki.append(f"{act} ({prob*100:.1f}%) -> {reason}")
    return wyniki

top_3_udf = F.udf(get_top_3_with_explanation, ArrayType(StringType()))

def rekomendacja_stacja_z_eksportem(wybrana_stacja):
    curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
    df_baza = spark.read.parquet(curated_s3_path)
    df_stacja = df_baza.filter(F.col("station_id") == wybrana_stacja).orderBy(F.col("timestamp").desc()).limit(1)
    
    if df_stacja.count() == 0:
        print(f"Brak danych dla stacji {wybrana_stacja}")
        return
        
    warunki = df_stacja.select("temperature", "wind_speed", "rain_mm", "cloud_cover").collect()[0]
    lokalizacja = lokalizacje_stacji.get(wybrana_stacja, "Nieznana lokalizacja")
    
    print(f"STACJA: {wybrana_stacja} ({lokalizacja})")
    print(f"POGODA: Temp: {warunki['temperature']}C | Wiatr: {warunki['wind_speed']} m/s | Opad: {warunki['rain_mm']} mm | Chmury: {warunki['cloud_cover']}%")
    
    pred = trained_model.transform(df_stacja)
    df_wynik = pred.withColumn("Rekomendacje", top_3_udf(F.col("probability")))
    
    print("\nAKTUALNE REKOMENDACJE:")
    for act in df_wynik.select("Rekomendacje").collect()[0][0]: print(f"- {act}")
    
    output_path = f"s3://michal-super-bucket-6767/analytical_output/recom_{wybrana_stacja}"
    df_wynik.select("station_id", "timestamp", "Rekomendacje").write.mode("append").json(output_path)

rekomendacja_stacja_z_eksportem("SOP_01")

scenarios_data = [
    (32.0, 2.0, 0.0, 5.0, "1. Fala upalow (Temp: 32C, Brak deszczu)"),
    (-10.0, 15.0, 0.0, 80.0, "2. Siarczysty mroz (Temp: -10C)"),
    (12.0, 5.0, 15.0, 100.0, "3. Ulewa i chlod (Temp: 12C, Opad: 15mm)"),
    (18.0, 12.0, 0.0, 40.0, "4. Bardzo wietrznie (Wiatr: 12m/s, Brak deszczu)")
]

schema_scenarios = StructType([
    StructField("temperature", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("rain_mm", DoubleType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("scenario_name", StringType(), True)
])

df_scenarios = spark.createDataFrame(scenarios_data, schema_scenarios)
pred_scenarios = trained_model.transform(df_scenarios)
df_scenarios_wynik = pred_scenarios.withColumn("Najlepszy_Wybor", top_3_udf(F.col("probability")))

results = df_scenarios_wynik.select("scenario_name", "Najlepszy_Wybor").collect()
for row in results:
    print(f"\n{row['scenario_name']}")
    for rec in row['Najlepszy_Wybor'][:3]:
        print(f"  * {rec}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

STACJA: SOP_01 (Sopot Molo)
POGODA: Temp: 17.9C | Wiatr: 9.08 m/s | Opad: 0.15 mm | Chmury: 39.0%

AKTUALNE REKOMENDACJE:
- Muzeum (99.6%) -> Lekki deszcz nie przeszkadza w zwiedzaniu pod dachem.
- Basen kryty (0.4%) -> Dobra opcja, gdy mocno pada, ale jest w miare cieplo.
- Jazda na rowerze (0.0%) -> Optymalna temperatura na rekreacje na zewnatrz.

1. Fala upalow (Temp: 32C, Brak deszczu)
  * Jazda na rowerze (94.8%) -> Optymalna temperatura na rekreacje na zewnatrz.
  * Spacer (4.7%) -> Rzesko i bez opadow, w sam raz na spacer.
  * Kitesurfing (0.5%) -> Swietne warunki wietrzne i brak opadow.

2. Siarczysty mroz (Temp: -10C)
  * Kitesurfing (99.2%) -> Swietne warunki wietrzne i brak opadow.
  * Spacer (0.5%) -> Rzesko i bez opadow, w sam raz na spacer.
  * Jazda na rowerze (0.3%) -> Optymalna temperatura na rekreacje na zewnatrz.

3. Ulewa i chlod (Temp: 12C, Opad: 15mm)
  * Muzeum (92.7%) -> Lekki deszcz nie przeszkadza w zwiedzaniu pod dachem.
  * Basen kryty (7.3%) -> Dobra opcja,

In [9]:
import pyspark.sql.functions as F

rf_model = trained_model.stages[2]
importances = rf_model.featureImportances.toArray()
features = ["Temperatura", "Predkosc wiatru", "Opady (mm)", "Zachmurzenie"]

print("WYKRES WAZNOSCI PARAMETROW POGODOWYCH W MODELU:")
print("-" * 65)
for feat, imp in sorted(zip(features, importances), key=lambda x: x[1], reverse=True):
    bar_length = int(imp * 40)
    bar = "-" * bar_length
    print(f"{feat:>16} | {bar} ({imp*100:.1f}%)")
print("-" * 65 + "\n")

print("TABELA: CZESTOTLIWOSC WYSTEPOWANIA ETYKIET W ZBIORZE:")
print("-" * 65)
df_rozklad = df_input.groupBy("activity").count().orderBy(F.col("count").desc())
df_rozklad.show(20, truncate=False)
print("="*50 + "\n")

print("OFICJALNY RAPORT METRYK MODELU ML")
print("="*50)
print(f"Algorytm klasyfikacji:  Random Forest (Las Losowy)")
print(f"Liczba drzew w lesie:   {rf_model.getNumTrees}")
print(f"Maksymalna glebokość:   {rf_model.getOrDefault('maxDepth')}")
print("-" * 50)
print(f"Calkowita dokladnosc (Accuracy):     {accuracy * 100:.2f}%")
print(f"Blad klasyfikacji (Error Rate):      {(1 - accuracy) * 100:.2f}%")
print("="*50)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

WYKRES WAZNOSCI PARAMETROW POGODOWYCH W MODELU:
-----------------------------------------------------------------
      Opady (mm) | ------------------------------ (75.0%)
 Predkosc wiatru | ------ (15.3%)
     Temperatura | --- (9.3%)
    Zachmurzenie |  (0.4%)
-----------------------------------------------------------------

TABELA: CZESTOTLIWOSC WYSTEPOWANIA ETYKIET W ZBIORZE:
-----------------------------------------------------------------
+----------------+-----+
|activity        |count|
+----------------+-----+
|Muzeum          |3384 |
|Jazda na rowerze|1020 |
|Kitesurfing     |500  |
|Spacer          |273  |
|Basen kryty     |23   |
+----------------+-----+


OFICJALNY RAPORT METRYK MODELU ML
Algorytm klasyfikacji:  Random Forest (Las Losowy)
Liczba drzew w lesie:   20
Maksymalna gleboko??:   5
--------------------------------------------------
Calkowita dokladnosc (Accuracy):     98.77%
Blad klasyfikacji (Error Rate):      1.23%